# Chapter 7: Telling Birds from Airplanes: Learning from Images
### Visual Classification, CIFAR-10, Preprocessing Pipelines, Softmax & CrossEntropy Losses, and Dense Layer Limitations

This companion notebook implements the complete pedagogical workflow of **Chapter 7** from *Deep Learning with PyTorch (2nd Edition)* by Eli Stevens, Luca Antiga, Thomas Viehmann, and Howard Huang.

#### Contents:
1. Setup & Environment Configuration
2. CIFAR-10 Dataset Ingestion & Inspection
3. Dataset Transforms & Channel-wise Normalization
4. Dataset Subsetting: Framing the Binary Problem (Birds vs Airplanes)
5. Baseline Fully Connected Classifier (`nn.Sequential` & Parameter Count)
6. Output Probability Calibration: Softmax, NLLLoss & CrossEntropyLoss
7. Mini-Batch Training with PyTorch `DataLoader`
8. The Complete Training & Validation Loop
9. Evaluating Model Performance & Visualizing Predictions
10. The Structural Limitations of Dense Networks: Testing Translation Invariance

In [ ]:
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

# Set deterministic random seeds
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch Version:     {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
print(f"Active Device:       {device}")

## 1. CIFAR-10 Dataset Ingestion & Inspection
The **CIFAR-10** benchmark contains 60,000 $32 \times 32$ color images across 10 classes. We download and instantiate the raw training and test partitions using `torchvision.datasets.CIFAR10`.

In [ ]:
data_path = Path('./data/cifar10')
data_path.mkdir(parents=True, exist_ok=True)

cifar10_raw = datasets.CIFAR10(root=str(data_path), train=True, download=True)
cifar10_raw_val = datasets.CIFAR10(root=str(data_path), train=False, download=True)

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 
               'dog', 'frog', 'horse', 'ship', 'truck']

print(f"Training set size:   {len(cifar10_raw)}")
print(f"Validation set size: {len(cifar10_raw_val)}")

# Sample inspection
img, label = cifar10_raw[99]
print(f"Sample type: {type(img)}, Size: {img.size}, Class: {label} ({class_names[label]})")

fig, axes = plt.subplots(1, 5, figsize=(12, 3))
for i in range(5):
    sample_img, sample_lbl = cifar10_raw[i]
    axes[i].imshow(sample_img)
    axes[i].set_title(class_names[sample_lbl])
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## 2. Dataset Transforms & Channel-wise Normalization
We convert PIL images into PyTorch CHW float32 tensors with `transforms.ToTensor()`, then calculate the per-channel empirical mean ($\mu$) and standard deviation ($\sigma$) across all 50,000 images.

In [ ]:
tensor_cifar10 = datasets.CIFAR10(
    root=str(data_path), 
    train=True, 
    download=False, 
    transform=transforms.ToTensor()
)

# Stack a subset of 10,000 images to calculate statistics efficiently
imgs = torch.stack([img_t for img_t, _ in tensor_cifar10], dim=3)
print(f"Stacked tensor shape: {imgs.shape}")  # (3, 32, 32, 50000)

# Flatten spatial dimensions: (3, 32 * 32 * 50000)
cifar_mean = imgs.view(3, -1).mean(dim=1)
cifar_std = imgs.view(3, -1).std(dim=1)

print(f"Per-channel Mean: {cifar_mean.tolist()}")
print(f"Per-channel Std:  {cifar_std.tolist()}")

# Define the complete normalized transform pipeline
transform_pipeline = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar_mean, std=cifar_std)
])

transformed_cifar10 = datasets.CIFAR10(
    root=str(data_path), 
    train=True, 
    download=False, 
    transform=transform_pipeline
)

transformed_cifar10_val = datasets.CIFAR10(
    root=str(data_path), 
    train=False, 
    download=False, 
    transform=transform_pipeline
)

## 3. Dataset Subsetting: Framing the Binary Problem (Birds vs Airplanes)
We filter the 10-class dataset down to two classes: `airplane` (index `0`) and `bird` (index `2`), mapping them to clean binary targets (`airplane -> 0`, `bird -> 1`).

In [ ]:
label_map = {0: 0, 2: 1}
class_names_binary = ['airplane', 'bird']

cifar2 = [
    (img, label_map[label]) 
    for img, label in transformed_cifar10 
    if label in [0, 2]
]

cifar2_val = [
    (img, label_map[label]) 
    for img, label in transformed_cifar10_val 
    if label in [0, 2]
]

print(f"cifar2 training samples:   {len(cifar2)}")
print(f"cifar2 validation samples: {len(cifar2_val)}")

## 4. Baseline Fully Connected Classifier
We construct a multi-layer perceptron (MLP) mapping flattened $3 \times 32 \times 32 = 3072$ pixel arrays into 512 hidden nodes followed by 2 class logits.

In [ ]:
n_in = 3072
n_hidden = 512
n_out = 2

model = nn.Sequential(
    nn.Linear(n_in, n_hidden),
    nn.Tanh(),
    nn.Linear(n_hidden, n_out),
    nn.LogSoftmax(dim=1)
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model Architecture:\n{model}")
print(f"\nTotal Trainable Parameters: {total_params:,}")

## 5. Mini-Batch Training with PyTorch `DataLoader`
We encapsulate our dataset into `DataLoader` instances with `batch_size=64` and shuffling enabled for training.

In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(cifar2, batch_size=64, shuffle=True)
val_loader = DataLoader(cifar2_val, batch_size=64, shuffle=False)

batch_imgs, batch_labels = next(iter(train_loader))
print(f"Batch Images Shape: {batch_imgs.shape}")
print(f"Batch Labels Shape: {batch_labels.shape}")

## 6. The Complete Training & Validation Loop
We train the network using stochastic gradient descent (`optim.SGD`) and Negative Log Likelihood Loss (`nn.NLLLoss`).

In [ ]:
def evaluate(model, loader, device='cpu'):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            batch_size = imgs.shape[0]
            outputs = model(imgs.view(batch_size, -1))
            _, preds = torch.max(outputs, dim=1)
            total += labels.shape[0]
            correct += int((preds == labels).sum())
    return correct / total

def train(model, train_loader, val_loader, n_epochs=30, lr=1e-2, device='cpu'):
    model = model.to(device)
    loss_fn = nn.NLLLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)
    
    history = {'train_loss': [], 'val_acc': []}
    
    for epoch in range(1, n_epochs + 1):
        model.train()
        running_loss = 0.0
        
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            batch_size = imgs.shape[0]
            
            outputs = model(imgs.view(batch_size, -1))
            loss = loss_fn(outputs, labels)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
        epoch_loss = running_loss / len(train_loader)
        val_acc = evaluate(model, val_loader, device=device)
        history['train_loss'].append(epoch_loss)
        history['val_acc'].append(val_acc)
        
        if epoch == 1 or epoch % 5 == 0:
            print(f"Epoch {epoch:2d}/{n_epochs:2d} | Train Loss: {epoch_loss:.4f} | Val Accuracy: {val_acc:.2%}")
            
    return history

# Execute training
history = train(model, train_loader, val_loader, n_epochs=20, lr=1e-2, device=device)

## 7. Evaluating Model Performance & Visualizing Predictions
We plot training loss and validation accuracy curves, then inspect sample predictions with their calibrated output probabilities.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], color='#b22222', lw=2)
ax1.set_title('Training Loss (NLL)')
ax1.set_xlabel('Epoch')
ax1.grid(True, alpha=0.3)

ax2.plot([acc * 100 for acc in history['val_acc']], color='#2e8b57', lw=2)
ax2.set_title('Validation Accuracy (%)')
ax2.set_xlabel('Epoch')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Display 5 sample predictions
model.eval()
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for i in range(5):
    img_t, true_lbl = cifar2_val[i]
    with torch.no_grad():
        log_p = model(img_t.view(1, -1).to(device))
        p = torch.exp(log_p).cpu().squeeze()
        pred_lbl = torch.argmax(p).item()
        
    # Denormalize image for visualization
    unnorm_img = img_t.clone()
    for c in range(3):
        unnorm_img[c] = unnorm_img[c] * cifar_std[c] + cifar_mean[c]
    unnorm_img = unnorm_img.permute(1, 2, 0).clamp(0, 1).numpy()
    
    axes[i].imshow(unnorm_img)
    axes[i].set_title(f"Pred: {class_names_binary[pred_lbl]} ({p[pred_lbl]:.2f})\nTrue: {class_names_binary[true_lbl]}",
                      color='green' if pred_lbl == true_lbl else 'red')
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## 8. Structural Limitations: Testing Translation Invariance
Dense layers are fundamentally not translation invariant. Let us test what happens to model confidence when we shift an image by just a few pixels.

In [ ]:
test_img, test_lbl = cifar2_val[0]

# Shift the image by 4 pixels to the right
shifted_img = torch.roll(test_img, shifts=4, dims=2)

with torch.no_grad():
    orig_p = torch.exp(model(test_img.view(1, -1).to(device))).cpu().squeeze()
    shifted_p = torch.exp(model(shifted_img.view(1, -1).to(device))).cpu().squeeze()

print("--- Translation Invariance Experiment ---")
print(f"Original Image Confidences: {class_names_binary[0]}={orig_p[0]:.3f}, {class_names_binary[1]}={orig_p[1]:.3f}")
print(f"Shifted Image Confidences:  {class_names_binary[0]}={shifted_p[0]:.3f}, {class_names_binary[1]}={shifted_p[1]:.3f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 3))
ax1.imshow(test_img.permute(1, 2, 0).clamp(0, 1))
ax1.set_title("Original Image")
ax1.axis('off')

ax2.imshow(shifted_img.permute(1, 2, 0).clamp(0, 1))
ax2.set_title("Shifted by 4 Pixels")
ax2.axis('off')
plt.tight_layout()
plt.show()